In [2]:
import ROOT
import pandas as pd
import numpy as np
import os
import sys
import ipynbname
from pathlib import Path

project_root = str(ipynbname.path().parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

from processing import SiphraAcquisition, MetadataLoader
from analysis import fit_peak_expbg

# Constants
BITS12 = 2**12
BITS9 = 2**9 # 512 typical number of bins used

# Energy emission peaks in MeV
Cs137_MeV = 0.662
Na22_MeV = [0.511, 1.275, 1.786]
Co60_MeV = [1.173, 1.332, 2,505]
Am241_MeV = 0.0595
# Background emission
K40_MeV = 1.460
Tl208_MEV = 2.614

colors = [ROOT.kRed, ROOT.kBlue, ROOT.kGreen, ROOT.kOrange, ROOT.kViolet, ROOT.kYellow, ROOT.kSpring, ROOT.kCyan,]

In [6]:
files = sorted((project_root/'data/260506').glob('SUBT_*'))
acqs = [SiphraAcquisition(file) for file in files]
#print(acqs)

bg_na_acq = acqs[0]
#led_na_acqs = [acqs[3], acqs[4], acqs[5], acqs[1], acqs[6], acqs[7], acqs[2]] #6 and 7 double peaks
led_na_acqs = [acqs[3], acqs[4], acqs[5], acqs[1], acqs[2]]
print(led_na_acqs)

bg_nosrc_acq = acqs[15]
led_nosrc_acqs = [acqs[13], acqs[12], acqs[10], acqs[11], acqs[9], acqs[8], acqs[14]]
print(led_nosrc_acqs)

[SIPHRAAcquisition(File: 'SUBT_04_LEDTesting_200Hz_77ns'), SIPHRAAcquisition(File: 'SUBT_05_LEDTesting_200Hz_78ns'), SIPHRAAcquisition(File: 'SUBT_06_LEDTesting_200Hz_79ns'), SIPHRAAcquisition(File: 'SUBT_02_LEDTesting_200Hz_80ns'), SIPHRAAcquisition(File: 'SUBT_03_LEDTesting_200Hz_83ns')]
[SIPHRAAcquisition(File: 'SUBT_14_LEDTesting_200Hz_78ns'), SIPHRAAcquisition(File: 'SUBT_13_LEDTesting_200Hz_79ns'), SIPHRAAcquisition(File: 'SUBT_11_LEDTesting_200Hz_80ns'), SIPHRAAcquisition(File: 'SUBT_12_LEDTesting_200Hz_80,5ns'), SIPHRAAcquisition(File: 'SUBT_10_LEDTesting_200Hz_81ns'), SIPHRAAcquisition(File: 'SUBT_09_LEDTesting_200Hz_81,5ns'), SIPHRAAcquisition(File: 'SUBT_15_LEDTesting_200Hz_82ns')]


In [9]:
# histograms
hists = {}
sources = []

bg = bg_nosrc_acq
filepath = bg.filepath.stem
src_bg = (MetadataLoader.load(bg.metadataFile)).source
print(src_bg)
sources.append(src_bg)
hist_bg = ROOT.TH1F(f"{src_bg} Background", "", BITS12, 0 , BITS12)
hist_bg.Fill(bg['s']/(len(bg.active_chs)-1))
hist_bg.GetXaxis().SetTitle("ADC channel number")
hist_bg.GetYaxis().SetTitle(r"Normalized counts")
hists[src_bg] = hist_bg


for sgnl in (led_nosrc_acqs):
    #bg = bg_na_acq
    filepath = sgnl.filepath.stem
    src = (MetadataLoader.load(sgnl.metadataFile)).source
    #src = src.split(" ", 1)[1]
    print(src)
    sources.append(src)
    # Signal and Background
    hist_sgnl = ROOT.TH1F(f"{src} Signal", "", BITS12, 0, BITS12)
    #hist_bg = ROOT.TH1F(f"{src} Background", "", BITS12, 0 , BITS12)
    hist_sgnl.Fill(sgnl['s']/(len(sgnl.active_chs)-1))
    #hist_bg.Fill(bg['s']/len(bg.active_chs))
    hist_sgnl.Scale(bg.exposure/sgnl.exposure)
    # Clean spectrum
    #hist_clean = hist_sgnl.Clone(f"{src} Clean")
    #hist_clean.Add(hist_bg, -1)
    #for hist in [hist_sgnl, hist_bg, hist_clean]:
        # hist.SetTitle(r"^{22}Na spectra - CI gain = 1/0.75 pC")
    hist_sgnl.GetXaxis().SetTitle("ADC channel number")
    hist_sgnl.GetYaxis().SetTitle(r"Normalized counts")
    hists[src] = hist_sgnl
    #hists[f"{src}_BG"] = hist_bg
    #hists[f"{src}_clean"] = hist_clean
    del hist_sgnl
del hist_bg

BG
LED200Hz78ns
LED200Hz79ns
LED200Hz80ns
LED200Hz80.5ns
LED200Hz81ns
LED200Hz82ns
LED200Hz82ns


Warning in <TROOT::Append>: Replacing existing TH1: LED200Hz82ns Signal (Potential memory leak).


In [10]:
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

Yinit = 0.82 # For stat boxes

cv = ROOT.TCanvas('cv', 'cv', 1600, 1200)

ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lg = ROOT.TLegend(0.48, 0.61, 0.75, 0.83)



print(hists)
print(sources)

for idx, (src) in enumerate(sources): 
    if idx == 0:
        sgnl = hists[src_bg]
        lg.AddEntry(sgnl, "Background ")
        sgnl.SetLineColor(colors[0])
        sgnl.Draw("hist")

    else:
        print(hists[src])
        sgnl = hists[src]
        lg.AddEntry(sgnl, src)
        sgnl.SetLineColor(colors[idx])
        sgnl.SetTitle(src)
        sgnl.Draw("sames hist")
    ROOT.gPad.Update()
    #for i, sp in enumerate([sgnl]):
    #st = sp.FindObject("stats")
    # print(type(st))
    #st.SetY1NDC(Yinit - idx*0.08)
    #st.SetY2NDC(idx*0.1)

    #lg.Draw()
    #cv.cd(idx+1).SetLogy()
    cv.SetLogy()
    #cv.Draw()

lg.Draw()
cv.Draw()


{'BG': <cppyy.gbl.TH1F object at 0x560848f46a70>, 'LED200Hz78ns': <cppyy.gbl.TH1F object at 0x5608482ec840>, 'LED200Hz79ns': <cppyy.gbl.TH1F object at 0x5608480cebf0>, 'LED200Hz80ns': <cppyy.gbl.TH1F object at 0x56084b7814a0>, 'LED200Hz80.5ns': <cppyy.gbl.TH1F object at 0x560848e39340>, 'LED200Hz81ns': <cppyy.gbl.TH1F object at 0x560848f4d520>, 'LED200Hz82ns': <cppyy.gbl.TH1F object at 0x560848e30960>}
['BG', 'LED200Hz78ns', 'LED200Hz79ns', 'LED200Hz80ns', 'LED200Hz80.5ns', 'LED200Hz81ns', 'LED200Hz82ns', 'LED200Hz82ns']
Name: LED200Hz78ns Signal Title:  NbinsX: 4096
Name: LED200Hz79ns Signal Title:  NbinsX: 4096
Name: LED200Hz80ns Signal Title:  NbinsX: 4096
Name: LED200Hz80.5ns Signal Title:  NbinsX: 4096
Name: LED200Hz81ns Signal Title:  NbinsX: 4096
Name: LED200Hz82ns Signal Title:  NbinsX: 4096
Name: LED200Hz82ns Signal Title: LED200Hz82ns NbinsX: 4096


In [10]:
from analysis import *

#Calibration based on the 3 peaks of the Na-22 spectrum


hist = hists[src_bg+'_BG']
energy_ranges = [(50, 150), (200, 300), (300, 400)]
energies = Na22_MeV

c_fit = calibration_fit(hist, energy_ranges, energies)
print(c_fit)

[0.005792614947738571, -0.07952629651149366]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      2579.87
NDf                       =           97
Edm                       =  4.31351e-10
NCalls                    =           73
Constant                  =      3873.91   +/-   11.4947     
Mean                      =      102.004   +/-   0.0525713   
Sigma                     =      22.1518   +/-   0.0520211    	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      406.848
NDf                       =           97
Edm                       =  8.16217e-09
NCalls                    =           77
Constant                  =       828.11   +/-   4.64882     
Mean                      =      233.689   +/-   0.248891    
Sigma                     =      33.6256   +/-   0.271489     	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2              

In [30]:
#Calibrating LED histograms based on Na22 calibration fit
hists_cal = []

hist_cal_na = calibrated_histogram(c_fit, bg_na_acq, BITS12)
hist_cal_na.Scale(1/bg_na_acq.exposure)
hists_cal.append(hist_cal_na)

for acq in led_na_acqs:
    hist_cal = calibrated_histogram(c_fit, acq, BITS12)
    hist_cal.Scale(1/acq.exposure)
    hists_cal.append(hist_cal)
    del hist_cal 


# Plot the LED histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1000, 800)
lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)

print(sources)
for idx, (src, hist_cal) in enumerate(zip(sources, hists_cal)):
    #print(hist_cal)
    hist_cal.SetLineColor(colors[idx])
    hist_cal.GetXaxis().SetTitle("Energy [MeV]")
    hist_cal.GetYaxis().SetTitle(r"Count rate [Hz]")
    lg1.AddEntry(hist_cal, src, "l")
    if idx == 0:
        hist_cal.Draw("hist")
    else:
        hist_cal.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()
    

#Calculate energy resolution
peaks = [(2.8, 3.6), (4.5, 5.6), (7.1, 8.4), (16.1, 17.4), (18.4, 19.6) ]
peak_energies = []
resolutions = []
    
for i in range(len(hists_cal)):
    hist = hists_cal[i]
    if i == 0:
        res = energy_resolution(hist, [(0.13, 0.6), (1.1, 1.5), (1.5 , 2.1)],  Na22_MeV)
    else:
        cal_fit=ROOT.TF1("cal_fit_" + str(i), "gaus", peaks[i-1][0], peaks[i-1][1])
        hist.Fit(cal_fit, "R+S")
        peak_energy = cal_fit.GetParameter(1)
        print(peak_energy)
        res = energy_resolution(hist, [peaks[i-1]], [peak_energy])
    resolutions.append(res)
print(resolutions[1:)

['Na-22', 'LED200Hz77ns', 'LED200Hz78ns', 'LED200Hz79ns', 'LED200Hz80ns', 'LED200Hz83ns']
3.2972386429097806
5.09160836864508
7.820399471999099
16.818614812817472
19.11771216376097
[[0.960385821042291, 0.3109275002789807, 0.47843769843718015], [0.12171995661949506], [0.0826125566409727], [0.06300367337229176], [0.03016893672537746], [0.021177619614539008]]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      10799.2
NDf                       =           78
Edm                       =  2.34439e-06
NCalls                    =          113
Constant                  =      9.77868   +/-   0.0737331   
Mean                      =      0.62049   +/-   0.00422203  
Sigma                     =     0.208389   +/-   0.0022388    	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      115.314
NDf                       =           66
Edm                       =  4.78926e-06
NCalls        

Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).


In [ ]:
#Calibration based on the 3 peaks of the Na-22 spectrum
hist = hists['Na22_clean']
energy_ranges = [(150, 200), (320, 420), (450, 520)]
energies = Na22_MeV

c_fit = calibration_fit(hist, energy_ranges, energies)

#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Na22 = calibrated_histogram(c_fit, acqs[4], BITS12)
hist_cal_bg_Na22.Scale(1/acqs[4].exposure) 
hist_cal_src_Na22 = calibrated_histogram(c_fit, acqs[3], BITS12)
hist_cal_src_Na22.Scale(1/acqs[3].exposure) 
hist_cal_clean_Na22 = hist_cal_src_Na22.Clone("Calibrated signal no BG")
hist_cal_clean_Na22.Add(hist_cal_bg_Na22, -1)


# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv2'):
    ROOT.gROOT.FindObject('cv2').Close()
cv2 = ROOT.TCanvas("cv2", "cv2", 1600, 800)
cv2.Divide(2,1)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv2.cd(1)
hist_cal_bg_Na22.SetLineColor(colors[0])
hist_cal_src_Na22.SetLineColor(colors[1])
hist_cal_src_Na22.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Na22.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Na22, "Background", "l")
lg1.AddEntry(hist_cal_src_Na22, r"Signal ^{22}Na", "l")
hist_cal_src_Na22.Draw("hist")
hist_cal_bg_Na22.Draw("sames hist")
lg1.Draw()
cv2.cd(1).SetLogy()
#hist_cal_bg.GetYaxis().SetRangeUser(0,1e4)


lg2 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv2.cd(2)
hist_cal_clean_Na22.SetLineColor(colors[2])
hist_cal_clean_Na22.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_clean_Na22.GetYaxis().SetTitle(r"Count rate [Hz]")
hist_cal_clean_Na22.Draw("hist")
lg2.AddEntry(hist_cal_clean_Na22, r"^{22}Na bg subtracted", "l")
lg2.Draw()
cv2.cd(2).SetLogy()

cv2.Draw()

#Calculate energy resolution
resolutions = []

print(res_Na22)

    
    


In [ ]:
#Calibration based on the 3 peaks of the Na-22 spectrum
hist = hists['Am-241_clean']
energy_ranges = [(0, 1), (5, 30)]
energies = [0, Am241_MeV]

c_fit = calibration_fit(hist, energy_ranges, energies)
#print(c_fit)

#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Am241  = calibrated_histogram(c_fit, acqs[8], BITS12)
hist_cal_bg_Am241.Scale(1/acqs[8].exposure) 
hist_cal_src_Am241 = calibrated_histogram(c_fit, acqs[7], BITS12)
hist_cal_src_Am241.Scale(1/acqs[7].exposure) 
hist_cal_clean_Am241 = hist_cal_src_Am241.Clone("Calibrated signal no BG")
hist_cal_clean_Am241.Add(hist_cal_bg_Am241, -1)


# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv4'):
    ROOT.gROOT.FindObject('cv4').Close()
cv4 = ROOT.TCanvas("cv4", "cv4", 1600, 800)
cv4.Divide(2,1)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv4.cd(1)
hist_cal_bg_Am241.SetLineColor(colors[0])
hist_cal_src_Am241.SetLineColor(colors[1])
lg1.AddEntry(hist_cal_bg_Am241, "Background", "l")
lg1.AddEntry(hist_cal_src_Am241, r"Signal ^{241}Am", "l")
hist_cal_src_Am241.Draw("hist")
hist_cal_bg_Am241.Draw("sames hist")
lg1.Draw()
cv4.cd(1).SetLogy()
#hist_cal_bg.GetYaxis().SetRangeUser(0,1e4)


lg2 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv4.cd(2)
hist_cal_clean_Am241.SetLineColor(colors[2])
hist_cal_clean_Am241.Draw("hist")
lg2.AddEntry(hist_cal_clean_Am241, r"^{241}Am bg subtracted", "l")
lg2.Draw()
cv4.cd(2).SetLogy()

cv4.Draw()

#Calculate energy resolution
res_Am241 = energy_resolution(hist_cal_clean_Am241, [(0.01, 0.09)],  [Am241_MeV])
print(res_Am241)

